In [165]:
import torch
import torch.nn as nn

In [166]:
text = "hello world hello machine learning hello pytorch machine learning is fun pytorch is powerful learning is fun"

In [167]:
# -------------------------
# 1. Character vocabulary
# -------------------------

chars = sorted(set(text))

stoi = {ch:i for i, ch in enumerate(chars) }
itos = {i:ch for ch, i in stoi.items()}

vocab_size = len(chars)
print("vocab_size:", vocab_size)
print("stoi:", stoi)

vocab_size: 20
stoi: {' ': 0, 'a': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'l': 9, 'm': 10, 'n': 11, 'o': 12, 'p': 13, 'r': 14, 's': 15, 't': 16, 'u': 17, 'w': 18, 'y': 19}


In [168]:
# -------------------------
# 2. Training data
# -------------------------
data = torch.tensor([stoi[ch] for ch in text])

x = data[:-1]   # current character
y = data[1:]    # next character

print("x:", x)
print("y:", y)


x: tensor([ 7,  4,  9,  9, 12,  0, 18, 12, 14,  9,  3,  0,  7,  4,  9,  9, 12,  0,
        10,  1,  2,  7,  8, 11,  4,  0,  9,  4,  1, 14, 11,  8, 11,  6,  0,  7,
         4,  9,  9, 12,  0, 13, 19, 16, 12, 14,  2,  7,  0, 10,  1,  2,  7,  8,
        11,  4,  0,  9,  4,  1, 14, 11,  8, 11,  6,  0,  8, 15,  0,  5, 17, 11,
         0, 13, 19, 16, 12, 14,  2,  7,  0,  8, 15,  0, 13, 12, 18,  4, 14,  5,
        17,  9,  0,  9,  4,  1, 14, 11,  8, 11,  6,  0,  8, 15,  0,  5, 17])
y: tensor([ 4,  9,  9, 12,  0, 18, 12, 14,  9,  3,  0,  7,  4,  9,  9, 12,  0, 10,
         1,  2,  7,  8, 11,  4,  0,  9,  4,  1, 14, 11,  8, 11,  6,  0,  7,  4,
         9,  9, 12,  0, 13, 19, 16, 12, 14,  2,  7,  0, 10,  1,  2,  7,  8, 11,
         4,  0,  9,  4,  1, 14, 11,  8, 11,  6,  0,  8, 15,  0,  5, 17, 11,  0,
        13, 19, 16, 12, 14,  2,  7,  0,  8, 15,  0, 13, 12, 18,  4, 14,  5, 17,
         9,  0,  9,  4,  1, 14, 11,  8, 11,  6,  0,  8, 15,  0,  5, 17, 11])


In [169]:
# -------------------------
# 3. Bigram model
# -------------------------
class BigramModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            512
        )

    def forward(self, x):
        return self.embedding(x)


model = BigramModel()


trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Trainable parameters: 10240


In [170]:
# -------------------------
# 4. Training
# -------------------------
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.01
)

for step in range(1000):

    logits = model(x)

    loss = nn.functional.cross_entropy(
        logits,
        y
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())


0 6.614960193634033
100 4.675942420959473
200 3.019681453704834
300 1.9664430618286133
400 1.4686189889907837
500 1.2558053731918335
600 1.162230134010315
700 1.1163787841796875
800 1.090296983718872
900 1.0736452341079712


In [172]:
# -------------------------
# 5. Generate
# -------------------------
idx = torch.tensor([stoi["f"]])
result = ""

logits = model(idx)

probs = torch.softmax(logits, dim=-1)

next_idx = torch.multinomial(probs, num_samples=1)

result = itos[next_idx.item()]

idx = next_idx

print(result)    

u
